# 4b - Naive Bayes

Questo notebook implementa e analizza un classificatore probabilistico Gaussian Naive Bayes per la previsione della variabile target `blueWins`.

Il modello viene addestrato esclusivamente sul Training Set standardizzato prodotto durante la fase di Data Preparation.

La valutazione durante la fase di addestramento viene effettuata mediante 5-Fold Stratified Cross-Validation, mantenendo il Test Set completamente separato per la successiva valutazione finale.

Gli obiettivi del notebook sono:

1. caricare il Training Set standardizzato;
2. implementare un classificatore Gaussian Naive Bayes;
3. valutarne l'accuracy mediante Cross-Validation;
4. descrivere l'assunzione di indipendenza condizionata alla base del modello;
5. addestrare il modello finale sull'intero Training Set;
6. esportare il classificatore in formato `.pkl`.

In [3]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB

In [4]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\mela\Desktop\predictLoL\predictLoL


## 1. Caricamento del Training Set

Vengono caricati il Training Set standardizzato e la relativa variabile target prodotti durante la fase di Data Preparation.

Il Training Set contiene esclusivamente le feature predittive, mentre `blueWins` rappresenta la variabile target:

- `blueWins = 0`: vittoria del Red Team;
- `blueWins = 1`: vittoria del Blue Team.

Il Test Set non viene utilizzato durante questa fase.

In [5]:
X_TRAIN_PATH = PROJECT_ROOT / "data" / "X_train_scaled.csv"
Y_TRAIN_PATH = PROJECT_ROOT / "data" / "y_train.csv"

assert X_TRAIN_PATH.exists(), f"File non trovato: {X_TRAIN_PATH}"
assert Y_TRAIN_PATH.exists(), f"File non trovato: {Y_TRAIN_PATH}"

X_train = pd.read_csv(X_TRAIN_PATH)
y_train = pd.read_csv(Y_TRAIN_PATH)["blueWins"]

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

X_train: (7903, 27)
y_train: (7903,)


In [6]:
print(f"Valori mancanti in X_train: {X_train.isna().sum().sum()}")
print(f"Valori mancanti in y_train: {y_train.isna().sum()}")

print("\nDistribuzione target:")
print(y_train.value_counts())

print("\ngameId presente:", "gameId" in X_train.columns)

Valori mancanti in X_train: 0
Valori mancanti in y_train: 0

Distribuzione target:
blueWins
0    3959
1    3944
Name: count, dtype: int64

gameId presente: False


## 2. Gaussian Naive Bayes

Naive Bayes è un classificatore probabilistico basato sul Teorema di Bayes.

Per una classe $Y$ e un vettore di feature $X$, il teorema può essere espresso come:

$$
P(Y \mid X) =
\frac{P(X \mid Y)P(Y)}
{P(X)}
$$

Nel presente problema, la classe corrisponde alla variabile `blueWins`, mentre $X$ rappresenta l'insieme delle informazioni relative allo stato della partita al minuto 10.

L'obiettivo consiste quindi nello stimare la probabilità a posteriori:

$$
P(\text{blueWins}=1 \mid X)
$$

e confrontarla con la probabilità relativa alla classe opposta.

### Assunzione di indipendenza condizionata

L'elemento caratterizzante del Naive Bayes è l'assunzione di indipendenza condizionata tra le feature una volta nota la classe.

Formalmente:

$$
P(x_1, x_2, \dots, x_n \mid Y)
=
\prod_{i=1}^{n} P(x_i \mid Y)
$$

Di conseguenza, la probabilità a posteriori della classe può essere calcolata, a meno di un fattore di normalizzazione, come:

$$
P(Y \mid x_1,\dots,x_n)
\propto
P(Y)
\prod_{i=1}^{n}P(x_i \mid Y)
$$

Questa semplificazione viene definita "naive" perché considera condizionalmente indipendenti feature che, nel problema reale, possono essere correlate.

Nel dataset di League of Legends, ad esempio, oro, esperienza, uccisioni e farming possono presentare relazioni tra loro. L'assunzione di indipendenza costituisce quindi un'approssimazione del fenomeno reale.

Tale assunzione permette tuttavia di semplificare notevolmente il calcolo della probabilità a posteriori, stimandola a partire dalle distribuzioni condizionate delle singole risorse di gioco.

### Assunzione Gaussiana

La variante `GaussianNB` viene utilizzata quando le feature sono numeriche continue.

Per ciascuna feature $x_i$ e classe $c$, viene stimata una distribuzione Gaussiana:

$$
P(x_i \mid Y=c)
=
\frac{1}{\sqrt{2\pi\sigma_{ic}^{2}}}
\exp\left(
-\frac{(x_i-\mu_{ic})^2}
{2\sigma_{ic}^{2}}
\right)
$$

dove:

- $\mu_{ic}$ rappresenta la media della feature $i$ per la classe $c$;
- $\sigma_{ic}^{2}$ rappresenta la relativa varianza.

Il classificatore combina quindi le probabilità condizionate delle singole feature con la probabilità a priori della classe per ottenere la probabilità a posteriori.

In [7]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 3. Addestramento e Cross-Validation

Viene inizializzato un classificatore `GaussianNB`.

Le prestazioni vengono stimate mediante 5-Fold Stratified Cross-Validation sul solo Training Set.

In ogni iterazione quattro fold vengono utilizzate per l'addestramento e una fold per la validazione. Il processo viene ripetuto cinque volte, in modo che ogni fold venga utilizzata una volta per la validazione.

L'accuracy media delle cinque iterazioni fornisce una stima delle prestazioni del modello sui dati non utilizzati nel relativo addestramento.

In [8]:
naive_bayes_model = GaussianNB()

naive_bayes_model

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [9]:
cv_scores = cross_val_score(
    naive_bayes_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Accuracy per ciascuna fold:")
print(cv_scores)

print(
    f"\nAccuracy media in Cross-Validation: "
    f"{cv_scores.mean():.4f}"
)

print(
    f"Deviazione standard accuracy: "
    f"{cv_scores.std():.4f}"
)

Accuracy per ciascuna fold:
[0.74383302 0.7311828  0.73371284 0.70886076 0.73227848]

Accuracy media in Cross-Validation: 0.7300
Deviazione standard accuracy: 0.0115


In [10]:
cv_results = pd.DataFrame({
    "fold": np.arange(1, len(cv_scores) + 1),
    "accuracy": cv_scores
})

cv_results

,fold,accuracy
0,1,0.743833
1,2,0.731183
2,3,0.733713
3,4,0.708861
4,5,0.732278


In [11]:
best_naive_bayes_model = GaussianNB()

best_naive_bayes_model.fit(
    X_train,
    y_train
)

print("Gaussian Naive Bayes addestrato sull'intero Training Set.")

Gaussian Naive Bayes addestrato sull'intero Training Set.


In [12]:
print("Classi:", best_naive_bayes_model.classes_)
print("Probabilità a priori:", best_naive_bayes_model.class_prior_)

Classi: [0 1]
Probabilità a priori: [0.50094901 0.49905099]


In [13]:
class_priors = pd.DataFrame({
    "classe": best_naive_bayes_model.classes_,
    "probabilita_a_priori": best_naive_bayes_model.class_prior_
})

class_priors

,classe,probabilita_a_priori
0,0,0.500949
1,1,0.499051


## 4. Salvataggio del modello

Dopo la valutazione mediante Cross-Validation, il classificatore Gaussian Naive Bayes viene addestrato sull'intero Training Set.

Il modello finale viene salvato nella cartella `models/` in formato `.pkl` mediante `joblib`, in modo da poter essere successivamente caricato senza ripetere l'addestramento.

In [14]:
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NAIVE_BAYES_MODEL_PATH = (
    MODELS_DIR / "naive_bayes_best.pkl"
)

joblib.dump(
    best_naive_bayes_model,
    NAIVE_BAYES_MODEL_PATH
)

print(
    f"Naive Bayes salvato in: "
    f"{NAIVE_BAYES_MODEL_PATH}"
)

Naive Bayes salvato in: c:\Users\mela\Desktop\predictLoL\predictLoL\models\naive_bayes_best.pkl


In [15]:
print("Controllo file salvato:")

print(
    f"{'OK' if NAIVE_BAYES_MODEL_PATH.exists() else 'ERRORE'} "
    f"- {NAIVE_BAYES_MODEL_PATH.name}"
)

Controllo file salvato:
OK - naive_bayes_best.pkl


## 5. Conclusioni

In questo notebook è stato implementato un classificatore Gaussian Naive Bayes per la previsione della vittoria del Blue Team.

Il modello è stato valutato esclusivamente sul Training Set mediante 5-Fold Stratified Cross-Validation, mantenendo il Test Set separato per la successiva valutazione finale.

Naive Bayes utilizza il Teorema di Bayes e semplifica il calcolo della probabilità a posteriori assumendo l'indipendenza condizionata delle feature rispetto alla classe. Tale assunzione è particolarmente forte nel contesto delle partite di League of Legends, poiché alcune risorse di gioco risultano naturalmente correlate, ma permette di costruire un modello probabilistico semplice e computazionalmente efficiente.

La variante Gaussian Naive Bayes modella le distribuzioni delle singole feature attraverso distribuzioni Gaussiane.

Il classificatore finale è stato addestrato sull'intero Training Set ed esportato nella cartella `models/` in formato `.pkl`.